# 05 Endpoint: `bigbox-demand-forecast` serving `@champion`, and the parity test

Why real time: the app's what-if (a planner changes the promotion, price or lead time of a
queue item) needs a fresh forecast for one row in well under a second; the batch table can't
answer a question nobody asked yet. The request carries the feature row (from the Lakebase
synced recommendations table), so the endpoint needs no feature lookup.

CPU, small, scale-to-zero. **Parity:** the same 500 holdout rows scored by the batch job
(`gold.forecast`) and by the endpoint must agree within 1e-6.

In [ ]:
import datetime as dt
import json
import time

import numpy as np
import pandas as pd
import yaml
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, EndpointTag, ServedEntityInput
from mlflow.tracking import MlflowClient
from pyspark.sql import functions as F

from bigbox_ml.features import FEATURES, KEYS

dbutils.widgets.text("catalog", "bigbox_fe_bar")
dbutils.widgets.text("endpoint", "bigbox-demand-forecast")
CAT = dbutils.widgets.get("catalog")
ENDPOINT = dbutils.widgets.get("endpoint")
MODEL = f"{CAT}.ml.demand_lgbm"
TOLERANCE = 1e-6

w = WorkspaceClient()
VERSION = (
    MlflowClient(registry_uri="databricks-uc").get_model_version_by_alias(MODEL, "champion").version
)
entity = ServedEntityInput(
    entity_name=MODEL,
    entity_version=str(VERSION),
    workload_size="Small",
    scale_to_zero_enabled=True,
)
existing = {e.name for e in w.serving_endpoints.list()}
t0 = time.time()
if ENDPOINT not in existing:
    w.serving_endpoints.create_and_wait(
        name=ENDPOINT,
        config=EndpointCoreConfigInput(name=ENDPOINT, served_entities=[entity]),
        tags=[EndpointTag(key="project", value="bigbox-stores-osa")],
        timeout=dt.timedelta(minutes=45),
    )
    action = "created"
else:
    served = w.serving_endpoints.get(ENDPOINT).config.served_entities or []
    if [(s.entity_name, s.entity_version) for s in served] != [(MODEL, str(VERSION))]:
        w.serving_endpoints.update_config_and_wait(
            name=ENDPOINT, served_entities=[entity], timeout=dt.timedelta(minutes=45)
        )
        action = "updated"
    else:
        action = "unchanged"
ep = w.serving_endpoints.get(ENDPOINT)
print(
    f"{ENDPOINT} {action} in {time.time() - t0:.0f} s: ready={ep.state.ready.value}, "
    f"config_update={ep.state.config_update.value}, serving "
    f"{[(s.entity_name, s.entity_version, s.workload_size) for s in ep.config.served_entities]}"
)

bigbox-demand-forecast created in 795 s: ready=READY, config_update=NOT_UPDATING, serving [('bigbox_fe_bar.ml.demand_lgbm', '1', 'Small')]


## Parity: 500 holdout rows, batch vs endpoint

In [ ]:
with open("../generator/config.yaml") as fh:
    end = yaml.safe_load(fh)["calendar"]["history_end"]
start = end - dt.timedelta(days=55)
rows = (
    spark.read.table(f"{CAT}.ml.store_sku_features")
    .where(F.col("date").between(F.lit(start), F.lit(end)))
    .join(
        spark.read.table(f"{CAT}.gold.forecast").select(*KEYS, "forecast_units", "model_version"),
        KEYS,
    )
    .orderBy(F.xxhash64(*KEYS), *KEYS)
    .limit(500)
    .toPandas()
)
assert (rows["model_version"] == str(VERSION)).all(), "gold.forecast was scored by another version"


def records(df: pd.DataFrame) -> list[dict]:
    return [
        {k: (None if pd.isna(v) else v) for k, v in r.items()}
        for r in df[FEATURES].to_dict(orient="records")
    ]


def query(recs):
    t = time.perf_counter()
    r = w.serving_endpoints.query(name=ENDPOINT, dataframe_records=recs)
    return np.asarray(r.predictions, dtype=float), (time.perf_counter() - t) * 1000


for attempt in range(10):  # scale-to-zero: the first call after idling can wait for a replica
    try:
        query(records(rows.head(1)))
        break
    except Exception as e:
        print(f"warm-up attempt {attempt + 1}: {type(e).__name__}: {str(e)[:120]}")
        time.sleep(30)
pred, ms = query(records(rows))
delta = np.abs(pred - rows["forecast_units"].to_numpy())
print(
    f"500 rows in one request ({ms:.0f} ms): max |batch - endpoint| = {delta.max():.3e} "
    f"(tolerance {TOLERANCE:g}) -> {'PASS' if delta.max() <= TOLERANCE else 'FAIL'}"
)
assert delta.max() <= TOLERANCE

500 rows in one request (106 ms): max |batch - endpoint| = 0.000e+00 (tolerance 1e-06) -> PASS


## Single-row latency (what the app's what-if sends)

In [ ]:
lat = []
for i in range(30):
    _, ms = query(records(rows.iloc[[i]]))
    lat.append(ms)
lat = np.array(lat)
print(
    f"30 single-row requests from a serverless notebook: p50 {np.percentile(lat, 50):.0f} ms, "
    f"p95 {np.percentile(lat, 95):.0f} ms, max {lat.max():.0f} ms"
)
example = records(rows.iloc[[0]])
out, ms = query(example)
print("request:", json.dumps({"dataframe_records": example})[:600])
print(
    f"response: {{'predictions': [{out[0]:.6f}]}} in {ms:.0f} ms; "
    f"batch value {rows['forecast_units'].iloc[0]:.6f}"
)

30 single-row requests from a serverless notebook: p50 69 ms, p95 98 ms, max 117 ms
request: {"dataframe_records": [{"lag_1": 0.0, "lag_7": 0.0, "lag_14": 0.0, "lag_28": 1.0, "mean_7": 0.14285714285714285, "mean_28": 0.42857142857142855, "std_28": 0.634126487474228, "dow_mean_4w": 0.25, "instock_rate_28": 0.42857142857142855, "stockout_share_7": 0.0, "stockout_share_28": 0.0, "censored_prev_day": 0.0, "online_share_28": 0.0, "category_trend": 0.9329611627123539, "dow": 0.0, "promo_discount": 0.0, "effective_price": 4.99, "unit_price": 4.99, "perishable": 0.0, "shelf_life_days": 0.0, "lead_time_days": 7.0, "case_pack": 12.0, "category": "Household", "format": "STANDARD"}]}
response: {'predictions': [0.403484]} in 65 ms; batch value 0.403484


In [ ]:
dbutils.notebook.exit(
    json.dumps(
        {
            "endpoint": ENDPOINT,
            "action": action,
            "model_version": str(VERSION),
            "ready": ep.state.ready.value,
            "parity_rows": int(len(rows)),
            "parity_max_abs_delta": float(delta.max()),
            "parity_pass": bool(delta.max() <= TOLERANCE),
            "latency_ms": {
                "p50": round(float(np.percentile(lat, 50)), 1),
                "p95": round(float(np.percentile(lat, 95)), 1),
            },
        }
    )
)

{"endpoint": "bigbox-demand-forecast", "action": "created", "model_version": "1", "ready": "READY", "parity_rows": 500, "parity_max_abs_delta": 0.0, "parity_pass": true, "latency_ms": {"p50": 69.2, "p95": 98.0}}